In [1]:
import pandas as pd
from libreco.data import DatasetPure
from libreco.algorithms import UserCF

Instructions for updating:
non-resource variables are not supported in the long term


In [4]:
train_ratings = pd.read_csv('processed_dataset/MovieLens-1M/ratings/ml_1m_train_movielens.csv')
val_ratings = pd.read_csv('processed_dataset/MovieLens-1M/ratings/ml_1m_val_movielens.csv')
test_ratings = pd.read_csv('processed_dataset/MovieLens-1M/ratings/ml_1m_test_movielens.csv')

movies = pd.read_csv('processed_dataset/MovieLens-1M/movies/movies_movielens.csv')

FileNotFoundError: [Errno 2] No such file or directory: 'processed_dataset/MovieLens-1M/ratings/ml_1m_train_movielens.csv'

In [ ]:
train_ratings.rename(columns={'user_id': 'user', 'item_id': 'item', 'rating': 'label', 'timestamp': 'time'},
                     inplace=True)
val_ratings.rename(columns={'user_id': 'user', 'item_id': 'item', 'rating': 'label', 'timestamp': 'time'}, inplace=True)
test_ratings.rename(columns={'user_id': 'user', 'item_id': 'item', 'rating': 'label', 'timestamp': 'time'},
                    inplace=True)

In [ ]:
train_data, data_info = DatasetPure.build_trainset(train_ratings)
eval_data = DatasetPure.build_evalset(val_ratings)
test_data = DatasetPure.build_testset(test_ratings)
print(data_info)

In [ ]:
train_ratings = train_ratings.sort_values(by='time')
val_ratings = val_ratings.sort_values(by='time')
test_ratings = test_ratings.sort_values(by='time')

In [6]:
user_cf = UserCF(task="ranking", data_info=data_info, k_sim=200, sim_type="cosine", mode='invert')

In [7]:
# Training the model
user_cf.fit(train_data, verbose=2, eval_data=eval_data, k=5,
            metrics=["loss", "roc_auc", "precision", "recall", "ndcg"], neg_sampling=True)

Training start time: 2024-11-08 21:40:33
Final block size and num: (6040, 1)
sim_matrix elapsed: 3.628s
sim_matrix, shape: (6040, 6040), num_elements: 33969896, density: 93.1151 %


eval_pointwise:   0%|          | 0/25 [00:00<?, ?it/s]

Detect 1 unknown interaction(s), position: [5602]
No common interaction or similar neighbor for user 0 and item 818, proceed with default prediction
No common interaction or similar neighbor for user 1 and item 109, proceed with default prediction
No common interaction or similar neighbor for user 1 and item 801, proceed with default prediction
No common interaction or similar neighbor for user 1 and item 1998, proceed with default prediction
No common interaction or similar neighbor for user 1 and item 808, proceed with default prediction
No common interaction or similar neighbor for user 3 and item 3510, proceed with default prediction


eval_pointwise:   4%|▍         | 1/25 [00:00<00:11,  2.03it/s]

Detect 1 unknown interaction(s), position: [5320]


...

Detect 2 unknown interaction(s), position: [1280, 3058]


eval_pointwise:  84%|████████▍ | 21/25 [00:10<00:01,  2.09it/s]

Detect 1 unknown interaction(s), position: [3300]


eval_listwise: 100%|██████████| 3020/3020 [07:42<00:00,  6.53it/s]

	 eval log_loss: 1.4797
	 eval roc_auc: 0.6617
	 eval precision@5: 0.0787
	 eval recall@5: 0.0411
	 eval ndcg@5: 0.1782


### Two-Tower Model

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import pytorch_lightning as pl
from sentence_transformers import SentenceTransformer
import numpy as np
from pytorch_lightning.callbacks import Callback
import pandas as pd

D:\Anaconda\lib\site-packages\sentence_transformers\cross_encoder\CrossEncoder.py:11: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm, trange
D:\Anaconda\lib\site-packages\transformers\utils\generic.py:311: UserWarning: torch.utils._pytree._register_pytree_node is deprecated. Please use torch.utils._pytree.register_pytree_node instead.
  torch.utils._pytree._register_pytree_node(


In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.cuda.get_device_name(0)

'NVIDIA GeForce RTX 3060 Laptop GPU'

In [4]:
# full_ratings = pd.read_csv('./processed_dataset/MovieLens-1M/ratings/ml_1m_full_movielens.csv')
full_ratings = pd.read_csv('./processed_dataset/MovieLens-1M/ratings/ratings_fulldata_movielens.csv')

train_ratings = pd.read_csv('./processed_dataset/MovieLens-1M/ratings/ratings_traindata_movielens.csv')
val_ratings = pd.read_csv('./processed_dataset/MovieLens-1M/ratings/ratings_valdata_movielens.csv')
test_ratings = pd.read_csv('./processed_dataset/MovieLens-1M/ratings/ratings_testdata_movielens.csv')

movies = pd.read_csv('./processed_dataset/MovieLens-1M/movies/movies_movielens_modified.csv')
users = pd.read_csv('./processed_dataset/MovieLens-1M/users/users_movielens.csv')

In [5]:
movies['movie_features'] = '[MOVIE_DETAIL] title: ' + movies['title'] + ' [SEP] genres: ' + movies['genres']

In [6]:
# Create a dictionary for fast lookup
movie_features_dict = movies.set_index('item_id')['movie_features'].to_dict()

# Create lists of user and item texts
item_texts = [movie_features_dict[movieId] for movieId in full_ratings['item_id'].unique()]

# Create a mapping from userId and movieId to indices
movie_id_to_idx = {movieId: idx for idx, movieId in enumerate(full_ratings['item_id'].unique())}

# Map userId and movieId in ratings_df to indices
train_ratings['movie_idx'] = train_ratings['item_id'].map(movie_id_to_idx)

# Map userId and movieId in ratings_val to indices
val_ratings['movie_idx'] = val_ratings['item_id'].map(movie_id_to_idx)

# Map userId and movieId in ratings_test to indices
test_ratings['movie_idx'] = test_ratings['item_id'].map(movie_id_to_idx)

# Extract user indices, item indices, and ratings
train_item_indices = torch.LongTensor(train_ratings['movie_idx'].values).to(device)
train_labels = torch.FloatTensor(train_ratings['rating'].values).to(device)

# Extract user indices, item indices, and ratings for validation
val_item_indices = torch.LongTensor(val_ratings['movie_idx'].values).to(device)
val_labels = torch.FloatTensor(val_ratings['rating'].values).to(device)

# Extract user indices, item indices, and ratings for test
test_item_indices = torch.LongTensor(test_ratings['movie_idx'].values).to(device)
test_labels = torch.FloatTensor(test_ratings['rating'].values).to(device)

In [7]:
class TwoTowerModel(pl.LightningModule):
    def __init__(self, user_model_name, item_model_name, embedding_size=384):
        super(TwoTowerModel, self).__init__()
        self.user_model = SentenceTransformer(user_model_name)
        self.item_model = SentenceTransformer(item_model_name)

        self.user_fc = nn.Linear(embedding_size, embedding_size)
        self.item_fc = nn.Linear(embedding_size, embedding_size)

        self.criterion = nn.MSELoss()
        self.epoch_losses = {'train_loss': [], 'val_loss': []}

    def forward(self, user_text, item_text):
        user_embedding = self.user_model.encode(user_text, convert_to_tensor=True).to(device)
        item_embedding = self.item_model.encode(item_text, convert_to_tensor=True).to(device)

        user_output = self.user_fc(user_embedding)
        item_output = self.item_fc(item_embedding)

        dot_product = torch.matmul(user_output.squeeze(), item_output.T)
        dot_product = 4 * torch.sigmoid(dot_product) + 1

        return dot_product

    def training_step(self, batch, batch_idx):
        users, items, ratings = batch

        items = [item_texts[i] for i in items.tolist()]

        preds = self(users, items)

        loss = self.criterion(preds, ratings)
        self.log('train_loss', loss)
        return loss

    def validation_step(self, batch, batch_idx):
        users, items, ratings = batch

        items = [item_texts[i] for i in items.tolist()]

        preds = self(users, items)

        loss = self.criterion(preds, ratings)
        self.log('val_loss', loss)
        return loss

    def configure_optimizers(self):
        return optim.Adam(self.parameters(), lr=1e-5)


class PrintLossesCallback(Callback):
    def on_train_epoch_end(self, trainer, pl_module):
        train_loss = trainer.callback_metrics.get('train_loss')
        if train_loss is not None:
            pl_module.epoch_losses['train_loss'].append(train_loss.item())
            print(f"Epoch {trainer.current_epoch + 1}: Train Loss: {train_loss.item()}")

    def on_validation_epoch_end(self, trainer, pl_module):
        val_loss = trainer.callback_metrics.get('val_loss')
        if val_loss is not None:
            pl_module.epoch_losses['val_loss'].append(val_loss.item())
            print(f"Epoch {trainer.current_epoch + 1}: Val Loss: {val_loss.item()}")

In [17]:
best_model_path = './lightning_logs/movies/paraphrase-MiniLM-L12-v2/not-binarized/history_5-epochs_lr-1e-5_(occu + gen ) (new format - only genre for movies) (with header tag)/checkpoints/epoch=4-step=62320.ckpt'
# best_model = TwoTowerModel.load_from_checkpoint(best_model_path, user_model_name='paraphrase-MiniLM-L6-v2', item_model_name='paraphrase-MiniLM-L6-v2').to(device)
best_model = TwoTowerModel.load_from_checkpoint(best_model_path, user_model_name='paraphrase-MiniLM-L12-v2',
                                                item_model_name='paraphrase-MiniLM-L12-v2').to(device)

D:\Anaconda\lib\site-packages\huggingface_hub\file_download.py:797: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
D:\Anaconda\lib\site-packages\transformers\utils\generic.py:311: UserWarning: torch.utils._pytree._register_pytree_node is deprecated. Please use torch.utils._pytree.register_pytree_node instead.
  torch.utils._pytree._register_pytree_node(


In [8]:
def generate_last_user_texts_with_history(users, movies, ratings):
    user_histories = {user_id: [] for user_id in users['user_id'].unique()}
    last_user_texts = {}

    # Convert relevant columns to dictionaries for faster access
    user_features_dict = users.set_index('user_id').to_dict('index')
    movie_titles_dict = movies.set_index('item_id')['genres'].to_dict()

    for _, row in ratings.iterrows():
        user_id = row['user_id']
        movie_id = row['item_id']

        # Get user features
        user = user_features_dict[user_id]
        user_features = f"[USER_PROFILE] occupation: {user['occupation']} [SEP] gender: {user['gender']}"

        # Append the user's history (only the last 3 movies)
        history_movies = [movie_titles_dict[mid] for mid in user_histories[user_id][-3:]]

        history_str = ", ".join(history_movies)

        if history_str:
            combined_features = f"{user_features} [SEP] genres: {history_str}"
        else:
            combined_features = f"{user_features}"

        # Update the dictionary to keep the last text for each user
        last_user_texts[user_id] = combined_features

        # Update the user history after generating combined features
        user_histories[user_id].append(movie_id)

    return last_user_texts


# Generate the last user texts for the validation data
val_last_user_texts = generate_last_user_texts_with_history(users, movies, val_ratings)

In [434]:
full_items_embeddings = torch.stack(
    [best_model.item_model.encode(item_text, convert_to_tensor=True) for item_text in item_texts]).to(device)

## Gradio Visualization

In [9]:
import gradio as gr
print(gr. __version__)

4.44.1


In [10]:
import pkg_resources

# List of libraries to check versions
libraries = [
    "gradio",
    # "libreco",
    "torch",
    "pytorch_lightning",
    "sentence_transformers",
    "pandas",
    "numpy",
]

# Get installed versions
versions = {lib: pkg_resources.get_distribution(lib).version for lib in libraries}

versions

{'gradio': '4.44.1',
 'torch': '2.2.2',
 'pytorch_lightning': '2.3.0',
 'sentence_transformers': '3.0.1',
 'pandas': '1.5.3',
 'numpy': '1.24.3'}

In [11]:
import libreco as llb
llb.__version__

'1.5.1'

In [12]:
def CFGenerator(user_id):
    k=10
    user_id = int(user_id)
    # Check if user_id exists in the users dataframe
    if user_id not in users["user_id"].values:
        return [f"User ID {user_id} does not exist in the database."]

    recommended_items = user_cf.recommend_user(user_id, n_rec=k, filter_consumed=True)
    movie_ids = recommended_items[user_id]
    movie_titles = []
    for movie_id in movie_ids:
        title_row = movies[movies["item_id"] == movie_id]
        if not title_row.empty:
            movie_titles.append(title_row["title"].values[0])
        else:
            movie_titles.append(f"Unknown Movie ID: {movie_id}")

    # Add ranks to the movie titles with proper newline formatting
    movie_titles_string = "\n".join([f"{i + 1}. {title}" for i, title in enumerate(movie_titles)])
    return movie_titles_string

In [13]:
def TTGenerator(input_data, input_method="User ID"):
    k=10
    try:
        if input_method == "User ID":
            # Validate user ID input
            try:
                user_id = int(input_data)
            except ValueError:
                return "Invalid User ID. Please enter a valid integer."

            # Generate recommendations using user ID
            movie_ids = get_top_n_items_with_history_unseen_items_gradio(
                best_model, user_id, n=k, input_type="userId"
            )

        elif input_method == "Manual Data Entry":
            # Validate user text input
            user_text = input_data.strip()
            if not user_text:
                return "No user data provided. Please enter valid user text."

            # Generate recommendations using manual user text
            movie_ids = get_top_n_items_with_history_unseen_items_gradio(
                best_model, user_text, n=k, input_type="userText"
            )

        else:
            return "Invalid input method. Please choose either 'User ID' or 'Manual Data Entry'."
        # Retrieve movie titles
        movie_titles = []
        for movie_id in movie_ids:
            title_row = movies[movies["item_id"] == movie_id]
            if not title_row.empty:
                movie_titles.append(title_row["title"].values[0])
            else:
                movie_titles.append(f"Unknown Movie ID: {movie_id}")

        # Add ranks to the movie titles with proper newline formatting
        movie_titles_string = "\n".join([f"{i + 1}. {title}" for i, title in enumerate(movie_titles)])
        return movie_titles_string

    except ValueError as e:
        return str(e)
    except Exception as e:
        return f"An error occurred: {str(e)}"

In [14]:
def get_top_n_items_with_history_unseen_items_gradio(model, user_input, n, input_type="userId"):
    # Ensure the model is in evaluation mode
    model.eval()
    userId = 0
    if input_type == "userId":
        # Handle userId input
        try:
            userId = int(user_input)
        except ValueError:
            raise ValueError("Invalid userId. Please provide a valid integer.")

        if userId not in val_last_user_texts:
            raise ValueError(f"User ID {userId} does not exist in the database.")

        # Get the user text for the given userId
        user_text = val_last_user_texts[userId]
    elif input_type == "userText":
        # Handle userText input
        user_text = user_input.strip()
        if not user_text:
            raise ValueError("Invalid user text. Please provide non-empty text.")
    else:
        raise ValueError("Invalid input type. Must be 'userId' or 'userText'.")

    # Encode the user text
    user_embedding = model.user_model.encode(user_text, convert_to_tensor=True).to(device)
    # Compute the scores (dot product between user embedding and each item embedding)
    user_output = model.user_fc(user_embedding).to(device)
    item_output = model.item_fc(full_items_embeddings).to(device)
    dot_product = torch.matmul(user_output, item_output.t()).squeeze()

    if input_type == "userId":
        # Get items the user has seen in the training and validation data
        seen_items_train = train_ratings[train_ratings['user_id'] == userId]['item_id'].values
        seen_items_val = val_ratings[val_ratings['user_id'] == userId]['item_id'].values
        seen_items = set(np.concatenate((seen_items_train, seen_items_val)))

        # Get the top n + len(seen_items) item indices and their scores
        top_n_scores, top_n_indices = torch.topk(dot_product, n + len(seen_items))

        # Map indices back to item IDs
        top_n_item_ids = [list(movie_id_to_idx.keys())[list(movie_id_to_idx.values()).index(idx.item())] for idx in
                          top_n_indices]

        # Filter out seen items
        unseen_top_n_item_ids = [item for item in top_n_item_ids if item not in seen_items]

        return unseen_top_n_item_ids[:n]

    else:
        top_n_scores, top_n_indices = torch.topk(dot_product, n)
        top_n_item_ids = [list(movie_id_to_idx.keys())[list(movie_id_to_idx.values()).index(idx.item())] for idx in
                          top_n_indices]

        return top_n_item_ids[:n]

In [15]:
def rrf_score(ranks, k=10):
    return sum([1 / (k + rank) for rank in ranks])

def combine_recommendations_with_rrf_with_weight_for_gradio(user_id, k=10, cf_weight=1, tt_weight=4, input_method="userId"):
    # Dictionary to hold the RRF scores
    combined_scores = {}

    # If input is "userId", only use Two-Tower recommendations
    if input_method == "Manual Data Entry":
        tt_recommended_item = get_top_n_items_with_history_unseen_items_gradio(
            best_model, user_id, n=k, input_type="userText"
        )

        # Calculate RRF scores from Two-Tower recommendations
        for rank, item in enumerate(tt_recommended_item, start=1):
            weighted_rank = rank * tt_weight  # Apply the weight to TT ranks
            if item not in combined_scores:
                combined_scores[item] = rrf_score([weighted_rank], k)
            else:
                combined_scores[item] += rrf_score([weighted_rank], k)

    # If input is "User ID", combine CF and TT recommendations
    else:
        # Get Collaborative Filtering recommendations
        user_id = int(user_id)
        cf_recommended_items = user_cf.recommend_user(user_id, n_rec=k, filter_consumed=True)

        # Assign ranks and calculate weighted RRF scores from CF recommendations
        for rank, item in enumerate(cf_recommended_items[user_id], start=1):
            weighted_rank = rank * cf_weight  # Apply the weight to CF ranks
            if item not in combined_scores:
                combined_scores[item] = rrf_score([weighted_rank], k)
            else:
                combined_scores[item] += rrf_score([weighted_rank], k)

        # Get Two-Tower recommendations
        tt_recommended_item = get_top_n_items_with_history_unseen_items_gradio(
            best_model, user_id, n=k, input_type="userId"
        )
        # print(movieTitleOutput(tt_recommended_item))

        # Assign ranks and calculate weighted RRF scores from TT recommendations
        for rank, item in enumerate(tt_recommended_item, start=1):
            weighted_rank = rank * tt_weight  # Apply the weight to TT ranks
            if item not in combined_scores:
                combined_scores[item] = rrf_score([weighted_rank], k)
            else:
                combined_scores[item] += rrf_score([weighted_rank], k)

    # Sort the items based on their RRF scores in descending order
    sorted_items = sorted(combined_scores.items(), key=lambda x: x[1], reverse=True)

    # Retrieve the top N item IDs
    top_items = [item for item, score in sorted_items[:k]]

    # Get the movie titles for the top items
    movie_titles = []
    for movie_id in top_items:
        title_row = movies[movies["item_id"] == movie_id]
        if not title_row.empty:
            movie_titles.append(title_row["title"].values[0])
        else:
            movie_titles.append(f"Unknown Movie ID: {movie_id}")

    # Add ranks to the movie titles with proper newline formatting
    movie_titles_string = "\n".join([f"{i + 1}. {title}" for i, title in enumerate(movie_titles)])

    return movie_titles_string

In [16]:
# Function to show user preferences dynamically
def show_user_preferences(user_id):
    try:
        user_id = int(user_id)  # Ensure user_id is an integer
        # Lookup user preferences from the DataFrame
        user_row = users[users["user_id"] == user_id]
        if not user_row.empty:
            preferences = (
                f"Occupation: {user_row['occupation'].values[0]}, "
                f"Age: {user_row['age'].values[0]}, "
                f"Gender: {user_row['gender'].values[0]}"
            )
            # Update the field to visible and return the preferences
            return gr.update(visible=True, value=preferences)
        else:
            # User not found
            return gr.update(visible=False, value="User not found!")
    except ValueError:
        # Handle invalid input
        return gr.update(visible=False, value="Invalid User ID!")

In [17]:
# def generate_last_user_texts_with_history_for_gradio(users, movies, ratings, user_id):
# # Initialize a dictionary to store user histories
#     user_histories = {user_id: [] for user_id in users['user_id'].unique()}
#
#     # Convert relevant columns to dictionaries for faster access
#     movie_titles_dict = movies.set_index('item_id')['title'].to_dict()
#
#     # Initialize a variable to store the combined features for the specified user
#     last_user_text = ""
#
#     for _, row in ratings.iterrows():
#         current_user_id = row['user_id']
#         movie_id = row['item_id']
#
#         # Skip processing if the current user is not the specified user
#         if current_user_id != user_id:
#             continue
#
#         # Append the user's history (only the last 3 movies)
#         history_movies = [movie_titles_dict[mid] for mid in user_histories[current_user_id][-3:]]
#
#         # Create a string of the user's history
#         history_str = ",\n".join(history_movies)
#
#         # Generate the combined features for the user
#         if history_str:
#             last_user_text = f"{history_str}"
#         else:
#             last_user_text = "No history available."
#
#         # Update the user history after generating combined features
#         user_histories[current_user_id].append(movie_id)
#
#     return last_user_text
# def generate_last_user_texts_with_history_for_gradio(users, movies, ratings, user_id):
#     # Filter ratings DataFrame to only include rows for the specified user
#     user_ratings = ratings[ratings['user_id'] == user_id]
#     if user_ratings.empty:
#         return "No history available."
#
#     # Convert relevant columns to dictionaries for faster access
#     movie_titles_dict = movies.set_index('item_id')['title'].to_dict()
#
#     # Get the last 3 movies watched by the user
#     last_3_movies = user_ratings.tail(3)['item_id'].tolist()
#     history_movies = [movie_titles_dict.get(mid, "Unknown Movie") for mid in last_3_movies]
#
#     # Create a string of the user's history
#     history_str = ",\n".join(history_movies)
#     return history_str if history_str else "No history available."

In [20]:
# import pandas as pd
# import os
#
# # Define the path to save the user history dataset
# USER_HISTORY_PATH = "user_history.csv"
#
# # Initialize the user history dataset
# if not os.path.exists(USER_HISTORY_PATH):
#     user_history_df = pd.DataFrame(columns=["user_id", "history"])
#     user_history_df.to_csv(USER_HISTORY_PATH, index=False)
# else:
#     user_history_df = pd.read_csv(USER_HISTORY_PATH)
#
# def update_user_history(users, movies, ratings, user_id):
#     # Load the current history dataset
#     global user_history_df
#     user_history_df = pd.read_csv(USER_HISTORY_PATH)
#
#     # Filter ratings for the given user
#     user_ratings = ratings[ratings['user_id'] == user_id]
#     if user_ratings.empty:
#         return "No history available."
#
#     # Get the last 3 movies watched by the user
#     movie_titles_dict = movies.set_index('item_id')['title'].to_dict()
#     last_3_movies = user_ratings.tail(3)['item_id'].tolist()
#     history_movies = [movie_titles_dict.get(mid, "Unknown Movie") for mid in last_3_movies]
#
#     # Store the history as a string
#     history_str = ", ".join(history_movies)
#
#     # Update or add the user's history in the dataset
#     if user_id in user_history_df['user_id'].values:
#         user_history_df.loc[user_history_df['user_id'] == user_id, 'history'] = history_str
#     else:
#         user_history_df = user_history_df.append({'user_id': user_id, 'history': history_str}, ignore_index=True)
#
#     # Save the updated history dataset
#     user_history_df.to_csv(USER_HISTORY_PATH, index=False)
#
#     return history_str

In [29]:
# def update_all_user_histories(users, movies, ratings):
#     # Load or initialize the user history dataset
#     global user_history_df
#     if not os.path.exists(USER_HISTORY_PATH):
#         user_history_df = pd.DataFrame(columns=["user_id", "history"])
#     else:
#         user_history_df = pd.read_csv(USER_HISTORY_PATH)
#
#     # Convert relevant columns for faster lookups
#     movie_titles_dict = movies.set_index('item_id')['title'].to_dict()
#
#     # Initialize a list to collect new rows
#     new_rows = []
#
#     # Iterate through all users in the dataset
#     for user_id in ratings['user_id'].unique():
#         # Filter ratings for the current user
#         user_ratings = ratings[ratings['user_id'] == user_id]
#         if user_ratings.empty:
#             history_str = "No history available."
#         else:
#             # Get the last 3 movies watched by the user
#             last_3_movies = user_ratings.tail(3)['item_id'].tolist()
#             history_movies = [movie_titles_dict.get(mid, "Unknown Movie") for mid in last_3_movies]
#
#             # Store the history as a string
#             history_str = " - ".join(history_movies)
#
#         # Check if the user already exists in the dataset
#         if user_id in user_history_df['user_id'].values:
#             user_history_df.loc[user_history_df['user_id'] == user_id, 'history'] = history_str
#         else:
#             # Collect new row for this user
#             new_rows.append({'user_id': user_id, 'history': history_str})
#
#     # Add new rows to the DataFrame using pd.concat
#     if new_rows:
#         new_rows_df = pd.DataFrame(new_rows)
#         user_history_df = pd.concat([user_history_df, new_rows_df], ignore_index=True)
#
#     # Save the updated history dataset
#     user_history_df.to_csv(USER_HISTORY_PATH, index=False)
#     print(f"User history updated for {len(ratings['user_id'].unique())} users.")


In [30]:
update_all_user_histories(users, movies, val_ratings)

User history updated for 6040 users.


In [31]:
# Example usage
user_id = 1  # Replace with the specific user ID you want to query
last_user_text_for_user = generate_last_user_texts_with_history_for_gradio(users, movies, val_ratings, user_id)
print(last_user_text_for_user)

Aladdin,
Toy Story,
Tarzan


In [37]:
def show_user_history(user_id):
    try:
        # Ensure user_id is an integer
        user_id = int(user_id)

        # Generate the last user text by calling the function
        user_last_text = generate_last_user_texts_with_history_for_gradio(users, movies, val_ratings, user_id)

        # Return the last user text
        return gr.update(visible=True, value=user_last_text)
    except ValueError:
        # Handle invalid input
        return gr.update(visible=False, value="Invalid User ID!")
# users_history = './user__movielens_history.csv'
#
# def show_user_history(user_id):
#     try:
#         # Ensure user_id is an integer
#         user_id = int(user_id)
#
#         # Load the current user history
#         global user_history_df
#         user_history_df = pd.read_csv(users_history)
#
#         # Check if the user's history exists
#         if user_id in user_history_df['user_id'].values:
#             user_history = user_history_df.loc[user_history_df['user_id'] == user_id, 'history'].values[0]
#         else:
#             user_history = "No history available."
#
#         return gr.update(visible=True, value=user_history)
#     except ValueError:
#         # Handle invalid input
#         return gr.update(visible=True, value="Invalid User ID!")


In [39]:
show_user_history(1)

{'value': 'Aladdin - Toy Story - Tarzan',
 '__type__': 'update',
 'visible': True}

In [41]:
# Create the Gradio app
with gr.Blocks() as demo:
    gr.Markdown("## Leveraging Large Language Models in Hybrid Recommendation Systems")
    # Add general instructions
    gr.Markdown(
        """
        ### Info:
        All three models use the MovieLens 1M dataset, which contains 6040 unique users(UserId:1,2,...,6040)
        1. The first model uses CF as the recommender model, which is the baseline of the project and cannot handle new users.("Cold-Start problem")
        2. The second model users the two-tower model architecture with the SBERT as its LLM. You can choose to get recommendations for new users as well.(Manual Data Entry).
        3. The third model is a hybrid model, combining the first two models using the RRF technique.
        """
    )
    # Row to hold two identical blocks
    with gr.Row(equal_height=True):
        with gr.Column():
            gr.Markdown("## Collaborative Filtering")  # Heading
            user_id_input = gr.Textbox(label="User ID", placeholder="Enter your ID here")
            user_preferences_cf_output = gr.Textbox(label="User Preferences", visible=False)
            user_history_sbert_output = gr.Textbox(label="User History", visible=False)
            cf_recommendations_output = gr.Textbox(label="Recommendations")
            cf_generate_button = gr.Button("Generate CF Recommendations")

            cf_generate_button.click(CFGenerator, inputs=[user_id_input], outputs=[cf_recommendations_output])
            user_id_input.change(show_user_preferences, inputs=[user_id_input], outputs=[user_preferences_cf_output])
            user_id_input.change(show_user_history, inputs=[user_id_input], outputs=[user_history_sbert_output])

        with gr.Column():
            gr.Markdown("## SBERT Two-Tower Model")  # Heading

            # Radio Button to Choose Input Method
            input_method_sbert = gr.Radio(
                choices=["User ID", "Manual Data Entry"],
                label="Choose Input Method",
                value="User ID"
            )

            # Fields for User ID Input
            user_id_input_sbert = gr.Textbox(label="User ID (SBERT)", visible=True, placeholder="Enter User ID")
            user_preferences_sbert_output = gr.Textbox(label="User Preferences", visible=False)
            user_history_sbert_output = gr.Textbox(label="User History", visible=False)

            # Fields for Manual Data Entry
            occupation_input = gr.Textbox(label="Occupation", visible=False, placeholder="Enter Occupation")
            gender_input = gr.Textbox(label="Gender", visible=False, placeholder="Enter Gender")
            favorite_genres_input = gr.Textbox(label="Favorite Genres", visible=False, placeholder="Enter Favorite Genres (comma-separated)")

            # Output field
            tt_recommendations_output = gr.Textbox(label="Recommendations", interactive=False)
            tt_generate_button = gr.Button("Generate LLM Recommendations")

            # Dynamically show the relevant input fields based on the selected method
            input_method_sbert.change(
                lambda method: (
                    gr.update(visible=method == "User ID"),  # User ID field
                    gr.update(visible=method == "Manual Data Entry"),  # Occupation field
                    gr.update(visible=method == "Manual Data Entry"),  # Gender field
                    gr.update(visible=method == "Manual Data Entry"),   # Favorite Genres field
                    gr.update(visible=method == "User ID"),  # User Preferences field
                    gr.update(visible=method == "User ID")   # User History field
                ),
                inputs=[input_method_sbert],
                outputs=[user_id_input_sbert,
                        occupation_input,
                        gender_input,
                        favorite_genres_input,
                        user_preferences_sbert_output,
                        user_history_sbert_output]
            )
            user_id_input_sbert.change(show_user_preferences, inputs=[user_id_input_sbert], outputs=[user_preferences_sbert_output])
            user_id_input_sbert.change(show_user_history, inputs=[user_id_input_sbert], outputs=[user_history_sbert_output])

            # Handle button click for recommendations
            def generate_recommendations(user_id, occupation, gender, favorite_genres, input_method):
                try:
                    # Determine input_data based on the selected method
                    if input_method == "User ID":
                        input_data = user_id
                    else:
                        # Combine manual data into a single structure for processing
                        input_data = (
                            f"[USER_PROFILE] occupation: {occupation.strip()} [SEP] "
                            f"gender: {gender.strip()} [SEP] "
                            f"genres: {favorite_genres.strip()}"
                        )

                    # Call TTGenerator with the appropriate input
                    return TTGenerator(input_data=input_data, input_method=input_method)
                except Exception as e:
                    return f"An error occurred: {str(e)}"

            # Connect button to the generate_recommendations function
            tt_generate_button.click(
                generate_recommendations,
                inputs=[user_id_input_sbert, occupation_input, gender_input, favorite_genres_input, input_method_sbert],
                outputs=[tt_recommendations_output]
            )
        # Row for Hybrid Model
    with gr.Row():
        with gr.Column():
            gr.Markdown("## Hybrid Model")  # Heading

            # Radio Button to Choose Input Method
            input_method_hybrid = gr.Radio(
                choices=["User ID", "Manual Data Entry"],
                label="Choose Input Method",
                value="User ID"
            )

            # Fields for User ID Input
            user_id_input_hybrid = gr.Textbox(label="User ID (Hybrid)", visible=True, placeholder="Enter User ID")
            user_preferences_hybrid_output = gr.Textbox(label="User Preferences", visible=False)
            user_history_hybrid_output = gr.Textbox(label="User Last 3 seen ", visible=False)
           # Fields for Manual Data Entry
            occupation_input_hybrid = gr.Textbox(label="Occupation", visible=False, placeholder="Enter Occupation")
            gender_input_hybrid = gr.Textbox(label="Gender", visible=False, placeholder="Enter Gender")
            favorite_genres_input_hybrid = gr.Textbox(label="Favorite Genres", visible=False, placeholder="Enter Favorite Genres (comma-separated)")

            # Output field
            hybrid_recommendations_output = gr.Textbox(label="Recommendations", interactive=False)
            hybrid_generate_button = gr.Button("Generate Hybrid Recommendations")

            # Dynamically show the relevant input fields based on the selected method
            input_method_hybrid.change(
                lambda method: (
                    gr.update(visible=method == "User ID"),  # User ID field
                    gr.update(visible=method == "Manual Data Entry"),  # Occupation field
                    gr.update(visible=method == "Manual Data Entry"),  # Gender field
                    gr.update(visible=method == "Manual Data Entry"),   # Favorite Genres field
                    gr.update(visible=method == "User ID"),  # User Preferences field
                    gr.update(visible=method == "User ID")   # User History field
                ),
                inputs=[input_method_hybrid],
                outputs=[user_id_input_hybrid, occupation_input_hybrid, gender_input_hybrid, favorite_genres_input_hybrid, user_preferences_hybrid_output, user_history_hybrid_output]
            )
            user_id_input_hybrid.change(show_user_preferences, inputs=[user_id_input_hybrid], outputs=[user_preferences_hybrid_output])
            user_id_input_hybrid.change(show_user_history, inputs=[user_id_input_hybrid], outputs=[user_history_hybrid_output])

            # Handle button click for hybrid recommendations
            def generate_hybrid_recommendations(user_id, occupation, gender, favorite_genres, input_method):
                try:
                    # Determine input_data based on the selected method
                    if input_method == "User ID":
                        input_data = user_id
                        return combine_recommendations_with_rrf_with_weight_for_gradio(
                            user_id=input_data, input_method="User ID"
                        )
                    else:
                        # Combine manual data into a single structure
                        input_data = (
                            f"[USER_PROFILE] occupation: {occupation.strip()} [SEP] "
                            f"gender: {gender.strip()} [SEP] "
                            f"genres: {favorite_genres.strip()}"
                        )
                        return combine_recommendations_with_rrf_with_weight_for_gradio(
                            user_id=input_data, input_method="Manual Data Entry"
                        )
                except Exception as e:
                    return f"An error occurred: {str(e)}"

            # Connect button to the generate_hybrid_recommendations function
            hybrid_generate_button.click(
                generate_hybrid_recommendations,
                inputs=[user_id_input_hybrid, occupation_input_hybrid, gender_input_hybrid, favorite_genres_input_hybrid, input_method_hybrid],
                outputs=[hybrid_recommendations_output]
            )
# Launch the app
demo.launch(share=True)

Running on local URL:  http://127.0.0.1:7860

Could not create share link. Please check your internet connection or our status page: https://status.gradio.app.
